# Build a SeisSol dynamic-rupture deck, from raw data

You bring **raw data and a mesh**. This notebook builds every gridded input, uses those
freshly built files to work out the physical parameters, and writes a folder you can run.

It is deliberately thin: all the logic lives in `deckbuild/`, so each section is a
PARAMETERS cell followed by `build` / `verify` / `plot`. Edit the parameters, Run All.

**Order is not cosmetic.** The stress closure needs `Sv(z)` from the material and the
friction zoning needs the thermal field, so **material runs first**. Running them out of
order is the likeliest way to pair a new velocity model with a stale stress field.

## [0] Setup and project

In [ ]:
# ---- PARAMETERS ----------------------------------------------------------------
PROJECT       = "demo_planar"    # a descriptor stem under projects/
RAW_DATA_DIR  = None             # where YOUR raw data lives.  None -> data/<PROJECT>/
                                 # e.g. "/data/my_fault/raw_exports"
REQUIRE_FILES = True             # False skips existence checks, never schema checks
# ----------------------------------------------------------------------------------
from pathlib import Path
from deckbuild.bootstrap import init
from deckbuild.config import Project

B = init(project=PROJECT, require_files=False)          # locate + path + reload
if RAW_DATA_DIR is None:
    cfg = Project.load(B.root / "projects" / f"{PROJECT}.yaml",
                       require_files=REQUIRE_FILES)
else:
    # Point the descriptor at YOUR data.  Every `path:` in the descriptor is resolved
    # against this directory, so the same descriptor works wherever the data sits.
    cfg = Project.load(B.root / "projects" / f"{PROJECT}.yaml",
                       require_files=REQUIRE_FILES, data_dir=Path(RAW_DATA_DIR))
print(cfg.summary())

In [ ]:
# What raw data did we actually find?
from pathlib import Path
for label, spec in (("orientation", cfg.raw.orientation), ("velocity", cfg.raw.velocity),
                    ("thermal", cfg.raw.thermal)):
    if spec is None:
        print(f"  {label:12} (none)"); continue
    n = ""
    if spec.path:
        hits = sorted(Path(cfg.data_dir).glob(spec.path)) if any(c in spec.path for c in "*?[") \
               else ([Path(cfg.resolve_path(spec.path))] if Path(cfg.resolve_path(spec.path)).exists() else [])
        n = f"  ({len(hits)} file(s))"
    print(f"  {label:12} kind={spec.kind:14}{n}")
print(f"  {'mesh':12} {cfg.resolve_path(cfg.mesh().path)}")

## [1] What we are building

```
RAW (you provide)        BUILT HERE                     CONSUMED BY
-----------------        ----------                     -----------
velocity model       ->  material nc               ->   deck; Vs gate; L_b
                          +-> plasticity nc        ->   deck (if plasticity on)
                          +-> Sv(z) profile        ->   STRESS closure
                          +-> Qp/Qs relation       ->   deck yaml (if Q on)
temperature / depth  ->  thermal nc or profile     ->   FRICTION zoning
stress orientation   ->  SHmax(x,y), R(x,y)        ->   STRESS orientation
fault mesh           ->  .puml.h5 + gates          ->   deck; all projections
                                  |
                                  v
              STRESS nc, FRICTION nc, rs_muw LuaMap
                                  |
                                  v
              PARAMETER DETERMINATION (reads the BAKED ncs, on the real mesh)
                                  |
                                  v
              DECK FOLDER (ncs + yamls + par + receivers + mesh)
```

Parameter determination happens **after** baking, never before: every read-out samples the
written NetCDF at the real fault facets, so what you tune against is what SeisSol reads.

In [ ]:
# Where EVERYTHING will land -- printed before anything is written.
RUN_TAG = "k1.7_case1_fz500"
OUT  = B.root / "outputs" / cfg.name / RUN_TAG
DECK = B.root / "decks" / f"{cfg.name}_{RUN_TAG}_nb"   # _nb: the notebook owns this deck;
                                                     # run_workflow.py writes _cli.
for sub in ("material", "stress", "friction"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)
print(f"artifacts -> {OUT}\ndeck      -> {DECK}")

## [2] Mesh — **ingest**, convert, gate

This notebook does **not** build a mesh. To build or improve one, open
`skills/code-mesh-build-improve/SKILL.md` with Claude; `MESHING.md` is the one-page map.
Here we ingest the mesh the descriptor points at, gate it (A–G), and snap the hypocentre
onto it.

In [ ]:
# ---- PARAMETERS ----------------------------------------------------------------
HYPOCENTER      = None    # None -> take it from the descriptor; or dict(lon=, lat=, depth_m=)
HYPO_SNAP_TOL_M = None    # None -> the descriptor's value
# ----------------------------------------------------------------------------------
from deckbuild.mesh import MeshStage
mesh_art = MeshStage().build(cfg, OUT)
print(f"ingested {mesh_art.path}")

In [ ]:
import dataclasses
from deckbuild.config import Hypocenter
from deckbuild.geometry import load_fault, snap_hypocenter

fault = load_fault(cfg.mesh(), cfg.data_dir, strike=cfg.strike)
hypo  = cfg.hypocenter() if HYPOCENTER is None else Hypocenter(**HYPOCENTER)
if HYPO_SNAP_TOL_M is not None:
    hypo = dataclasses.replace(hypo, snap_tol_m=HYPO_SNAP_TOL_M)
snap = snap_hypocenter(hypo, fault, cfg.strike, cfg.crs, gate_bands=cfg.gate_bands)
snap.report.print()

In [ ]:
# Plot requested vs snapped ON the fault.  No rule can tell a correct snap from one onto
# the WRONG STRAND -- that is why this plot exists.
import matplotlib.pyplot as plt, numpy as np
s = fault.s_km(cfg.strike); d = fault.depth_m / 1000.0
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(s, d, s=6, c="0.8", label="fault facets")
ax.scatter([snap.s_km], [snap.depth_m / 1000.0], marker="*", s=260,
           c="crimson", zorder=5, label=f"snapped ({snap.distance_m:.0f} m)")
for b in cfg.gate_bands:
    ax.axvspan(b.s_start_km, b.s_end_km, alpha=.15, color="tab:orange")
    ax.text(0.5*(b.s_start_km+b.s_end_km), d.max()*0.95, b.name, ha="center", fontsize=8)
ax.invert_yaxis(); ax.set_xlabel("s along strike (km)"); ax.set_ylabel("depth (km)")
ax.legend(); ax.set_title("hypocentre placement"); plt.tight_layout(); plt.show()

## [3] Material — raw velocity → nc, plasticity, Sv(z)

**Runs first**, because stress needs its `Sv(z)` and friction may need its thermal field.

`plastCo = 1e-4 mu` is LINEAR in mu, so ASAGI's interpolation of the cohesion stays exactly
consistent with the shear modulus each element sees. `bulkFriction` is `tan(phi)` — the
COEFFICIENT, not degrees; SeisSol computes `atan()` of it.

In [ ]:
# ---- PARAMETERS ----------------------------------------------------------------
# The material itself ({rho, mu, lambda}) is ALWAYS built -- SeisSol cannot run without
# it.  The next two are ADVANCED, OPT-IN physics: both default OFF, and turning either on
# changes what SeisSol computes and what binary you need.
WITH_PLASTICITY  = False   # off-fault Drucker-Prager yielding (Roten et al. 2014).
                           #   ON  -> writes a plasticity nc; sets Plasticity = 1.
                           #   Costs run time and DISSIPATES energy: on SAFS it lowered
                           #   Mw and, alone, blocked the San Gorgonio gate.
WITH_ATTENUATION = False   # intrinsic (anelastic) Q, from the CVM's own Vs.
                           #   ON  -> writes Qp/Qs settings into the deck.
                           #   *** REQUIRES a VISCOELASTIC SeisSol build.  EQUATIONS=
                           #   viscoelastic2 is a BUILD-time choice, not a runtime flag;
                           #   an elastic binary CANNOT read the resulting deck. ***

# --- only read when WITH_PLASTICITY is True ---
PHI_SOFT, PHI_HARD = 35.0, 45.0   # friction angle, deg: soft rock / hard rock.
                                  # 35/45 is Roten et al. (2014).  The SAFS production
                                  # decks use 30/40 -- neither is a silent default.
VS_THRESHOLD       = 2500.0       # m/s: the Vs that splits soft from hard
COHESION_FACTOR    = 1.0e-4       # c = factor * mu   (Roten cohesion model 3)

# --- only read when WITH_ATTENUATION is True ---
Q_FREQ_CENTRAL     = 0.5          # Hz: Q is held ~constant over a band centred here.
Q_FREQ_RATIO       = 100.0        # the band is FreqCentral * sqrt(ratio) either side;
                                  # OUTSIDE it Q -> infinity, i.e. the medium goes elastic.
# ----------------------------------------------------------------------------------
from deckbuild.material import AttenuationSpec, MaterialStage, PlasticitySpec
mstage = MaterialStage()
mat = mstage.build(
    cfg, OUT / "material",
    plasticity=PlasticitySpec(PHI_SOFT, PHI_HARD, VS_THRESHOLD, COHESION_FACTOR)
              if WITH_PLASTICITY else None,
    attenuation=AttenuationSpec(freq_central=Q_FREQ_CENTRAL, freq_ratio=Q_FREQ_RATIO)
              if WITH_ATTENUATION else None)
print(f"plasticity : {'ON  -> ' + Path(mat.plasticity.path).name if mat.plasticity else 'off'}")
print(f"attenuation: {'ON  (needs a viscoelastic build!)' if mat.attenuation else 'off'}")
mstage.verify(cfg, mat, plasticity_artifact=mat.plasticity).print()

In [ ]:
import numpy as np, matplotlib.pyplot as plt
svp = np.load(mat.sv_profile.path)   # NOT `d`: that is the fault depth from [2]
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(svp["sv_eff_mpa"], svp["depth_m"] / 1000.0)
ax[0].invert_yaxis(); ax[0].set_xlabel("Sv_eff (MPa)"); ax[0].set_ylabel("depth (km)")
ax[0].set_title("vertical effective stress (SEA-LEVEL referenced)")
from deckbuild.asagi import read_asagi
_, _, z, f, _ = read_asagi(mat.material.path)
ax[1].plot(np.sqrt(f["mu"] / f["rho"]).reshape(len(z), -1).mean(axis=1), -np.asarray(z)/1000)
ax[1].invert_yaxis(); ax[1].set_xlabel("Vs (m/s)"); ax[1].set_title("velocity model")
plt.tight_layout(); plt.show()

## [4] Stress — orientation × Sv × closure k

Andersonian C1: `sig2 = Sv_eff` (**vertical** — a strike-slip assumption),
`sig3 = Sv_eff/((1-R)k + R)`, `sig1 = k sig3`. Compression-**negative** Pa in the nc.

`k` is a column property, so a graded design is the same code path as a scalar. The
shallow freeze makes every node above its depth carry its own column's stress evaluated
*at* that depth — without it the trace band sits under 1 MPa and slides all run.

In [ ]:
# ---- PARAMETERS ----------------------------------------------------------------
K_VALUES        = [1.7]     # one k per along-strike region
K_BOUNDARIES    = []        # exactly len(K_VALUES)-1 smoothstep windows, e.g. [(150,160)]
FREEZE_ABOVE_M  = 500.0     # 0 disables the shallow freeze
# ----------------------------------------------------------------------------------
from deckbuild.stress import KDesign, StressStage
sstage = StressStage()
kdes = KDesign(k_values=tuple(K_VALUES),
               boundaries_s_km=tuple(tuple(b) for b in K_BOUNDARIES))
stress = sstage.build(cfg, OUT / "stress", sv_profile=mat.sv_profile, design=kdes,
                      freeze_above_depth_m=FREEZE_ABOVE_M)
sstage.verify(cfg, stress, design=kdes).print()

In [ ]:
# On-fault stress, sampled from the WRITTEN nc at the real facets.
from deckbuild.stage_f import project_stress_onto_fault
sn, tau, mu = project_stress_onto_fault(stress.path, fault)
fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
for a_, v, t in zip(ax, (sn, tau, mu), ("sigma_n (MPa)", "tau_0 (MPa)", "mu_app")):
    sc = a_.scatter(s, d, c=v, s=8, cmap="viridis"); plt.colorbar(sc, ax=a_)
    a_.invert_yaxis(); a_.set_title(t); a_.set_xlabel("s (km)")
ax[0].set_ylabel("depth (km)"); plt.tight_layout(); plt.show()
print(f"min sigma_n {np.nanmin(sn):.2f} MPa | min tau_0 {np.nanmin(tau):.3f} MPa | "
      f"max mu_app {np.nanmax(mu):.3f}")

## [5] Friction — a(T), V_w(T), and the graded `f_w` LuaMap

`rs_b` is a **scalar**: SeisSol v1.1.3 FL=103 never reads it spatially.

Spatial `rs_muw` needs SeisSol **newer than v1.3.2** — v1.1.3 accepts the YAML and
*silently ignores it*, so the run completes and is wrong. Check before you build.

### CASE 1 vs CASE 2 — what actually differs

Both map temperature to `a - b` (the rate-and-state stability parameter) and to `V_w` (the
strong-rate-weakening slip rate). They differ **only in the shallow zone**, and that one
difference changes the whole design.

| | CASE 1 — three zones | CASE 2 — two zones |
|:--|:--|:--|
| `T <= 50 °C` (shallow) | `a-b = +0.004` → velocity **STRENGTHENING** | `a-b = -0.004` → velocity **WEAKENING** |
| `50 – 150 °C` | ramps through 0 at 100 °C | still weakening |
| `150 – 300 °C` (seismogenic) | `a-b = -0.004` → weakening | `a-b = -0.004` → weakening |
| `T >= 300 °C` (deep) | ramps back up, 0 at 350 °C → strengthening | identical |
| the trace | **stable**: a shallow VS lid resists slip | **unstable**: VW all the way to the surface |

`a - b < 0` is velocity-weakening — slip can accelerate, so rupture nucleates and
propagates. `a - b > 0` is velocity-strengthening — the fault resists, so it acts as a
barrier.

**Why it matters for you:** CASE 2 has no shallow lid, so ruptures reach the free surface
and produce far more slip at the trace. A `k` or `f_w` calibration made on CASE 1 does
**not** transfer to CASE 2 — the SAFS work found this the hard way. `V_w` is the same in
both: 0.05 m/s up to 350 °C, ramping to 1000 m/s by 400 °C (which switches strong
weakening off in the deep VS zone).

In [ ]:
# ---- PARAMETERS ----------------------------------------------------------------
CASE          = 1     # which thermal zoning to use -- see the table above.
                      #   1 = three zones: VS shallow / VW seismogenic / VS deep
                      #   2 = two zones:   VW from the SURFACE down to the deep VS
FW_VALUES     = [0.0, 0.05]     # the strong-rate-weakening FLOOR f_w, one per region,
                                # ordered along increasing s.  Lower f_w = weaker at high
                                # slip rate = LARGER dynamic stress drop.
FW_BOUNDARIES = [(10.0, 20.0)]  # exactly len(FW_VALUES)-1 smoothstep windows, in s (km)
F0            = 0.6   # the REFERENCE friction coefficient of the rate-and-state law:
                      # the steady-state friction at the reference slip rate RS_sr0.
                      # It is the ceiling f_w is measured against -- gate F0 requires
                      # 0 <= f_w < f0 in every region, because f_w is the value friction
                      # DECAYS TO at high slip rate, and decaying UP to or past the
                      # reference would mean the fault strengthens when it slips fast.
                      # 0.6 is the Byerlee-like value used throughout the SAFS decks.
SEISSOL_SRC   = None  # path to your SeisSol tree, to check spatial rs_muw support
# ----------------------------------------------------------------------------------
from deckbuild.friction import (FrictionStage, FwDesign,
                                check_seissol_supports_spatial_muw, nucleation_lua)
check_seissol_supports_spatial_muw(SEISSOL_SRC).print()

In [ ]:
fstage = FrictionStage()
friction = fstage.build(cfg, OUT / "friction", case=CASE)
fstage.verify(cfg, friction).print()

fwdes = FwDesign(fw_values=tuple(FW_VALUES),
                 boundaries_s_km=tuple(tuple(b) for b in FW_BOUNDARIES), f0=F0)
fw = fstage.build_fw_map(cfg, OUT / "friction", fwdes)
fstage.verify_fw_map(cfg, fw, fwdes).print()

In [ ]:
from deckbuild.friction import fw_profile_1d
ss = np.linspace(float(s.min()) - 5, float(s.max()) + 5, 600)
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(ss, fw_profile_1d(ss, fwdes), lw=2)
ax.axvline(snap.s_km, color="crimson", ls="--", label="hypocentre")
for b in cfg.gate_bands:
    ax.axvspan(b.s_start_km, b.s_end_km, alpha=.15, color="tab:orange")
ax.set_xlabel("s along strike (km)"); ax.set_ylabel("f_w"); ax.legend()
ax.set_title("graded strong-rate-weakening floor"); plt.tight_layout(); plt.show()

## [6] Assemble the deck + pre-flight P1–P8

The pre-flight reads the assembled deck's live `file:` targets — it tests what SeisSol
will actually read, not what is in memory.

`P6` is a **warning by design**: kappa is necessary, not sufficient. A design passed this
screen and still arrested at s ≈ 60 km on the cluster.

In [ ]:
# ---- PARAMETERS ----------------------------------------------------------------
MU_S          = 0.6        # static strength, for the t=0 pre-slip check
END_TIME_S    = 20.0
NUC_RADIUS_M  = 2000.0
NUC_AMP_MPA   = 75.0
DECK_NOTES    = ""         # preserved verbatim across regeneration
# ----------------------------------------------------------------------------------
from deckbuild.deck import DeckSpec, DeckStage, resolve_paths
# z = -1 m, NEVER 0: SeisSol v1.1.3 silently DROPS a receiver at the free surface.
receivers = np.array([[cfg.stress_box.xmin + 5000.0 * i,
                       cfg.stress_box.ymin + 5000.0 * i, -1.0] for i in range(1, 6)])
arts = {"material": mat.material, "stress": stress,
        "friction": friction, "mesh": mesh_art}
if mat.plasticity is not None:            # only when WITH_PLASTICITY
    arts["plasticity"] = mat.plasticity
spec = DeckSpec(prefix=f"{cfg.name}_",
                plasticity=mat.plasticity is not None,
                attenuation=mat.attenuation,
                receivers=receivers, mu_s=MU_S, end_time_s=END_TIME_S,
                rs_muw_lua=Path(fw.path).read_text(),
                nucleation_lua=nucleation_lua(snap, NUC_RADIUS_M, NUC_AMP_MPA),
                notes=DECK_NOTES)
for k, e in resolve_paths(DECK, arts, spec).items():
    print(f"  {k:12} -> {e['filename']:44} (read by {e['referenced_by']})")

In [ ]:
dstage = DeckStage()
deck = dstage.assemble(cfg, DECK, arts, spec, overwrite=True)
dstage.preflight(cfg, deck, spec=spec).print()
print(f"\ndeck: {deck}")
for f in sorted(deck.iterdir()):
    print(f"  {f.name:52} {f.stat().st_size/1024:9.0f} KiB")

## [7] Design read-outs

What the design actually does, read off the baked files.

In [ ]:
lo, hi = cfg.physics.seis_band_km
band = (d >= lo) & (d <= hi)
dtau = mu[band] * sn[band]
print(f"seismogenic band {lo}-{hi} km: {int(band.sum())} facets")
print(f"  dynamic stress drop proxy  min {np.nanmin(dtau):6.2f}  "
      f"median {np.nanmedian(dtau):6.2f}  max {np.nanmax(dtau):6.2f} MPa")
print(f"  L_b = mu*Dc/(b*sigma_n) at the hypocentre: "
      f"{cfg.physics.mu_shear_pa*cfg.physics.dc_m/(cfg.physics.rs_b*sn[snap.facet_index]*1e6):.0f} m")
print(f"  hypocentre sigma_n {sn[snap.facet_index]:.2f} MPa, "
      f"tau_0 {tau[snap.facet_index]:.2f} MPa, mu_app {mu[snap.facet_index]:.3f}")

---

### Retargeting to your own fault

Copy `projects/demo_planar.yaml`, edit it, and set `PROJECT` in section [0]. You supply the
fault mesh (built with the skill), a velocity model, a stress orientation, a friction
parameterisation, a hypocentre, and the strike frame + CRS. You keep every `physics:`
default. You decide one thing yourself: **the tectonic regime** — the Andersonian closure
assumes `sigma2` is vertical, which is a strike-slip assumption.

Go deeper on a stage: the legacy deep-dive notebooks
(`combined_stress_friction_check`, `combined_graded_fw_check`, `initial_state_variable`,
`plasticity_roten2014`) remain the reference for the design side.